# Hull Tactical Market Prediction - Kaggle Submission

**Estrategia**: Ensemble de LightGBM + XGBoost + CatBoost con ingeniería de características avanzada

**Características clave**:
- Lags temporales (1, 2, 3, 5 períodos)
- Rolling statistics (medias móviles, volatilidad)
- Ratios entre características
- Momentum indicators (ROC, RSI)
- Características temporales cíclicas
- Validación temporal estricta

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.feature_selection import SelectFromModel

print("📚 Librerías importadas")

In [ ]:
def adjusted_sharpe_ratio(y_true, y_pred, max_weight=6.0):
    """Métrica de evaluación de la competencia"""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_pred = np.clip(y_pred, -max_weight, max_weight)
    
    strategy_returns = y_true * y_pred
    mean_return = np.mean(strategy_returns)
    std_return = np.std(strategy_returns)
    
    if std_return == 0:
        return 0.0
    
    sharpe = mean_return / std_return
    volatility_penalty = 1.0 / (1.0 + std_return)
    
    return sharpe * volatility_penalty

print("✅ Función de evaluación definida")

In [ ]:
def create_features(df):
    """Ingeniería de características optimizada"""
    df_feat = df.copy()
    feature_cols = [col for col in df.columns if col.startswith('feature_')]
    
    # 1. Lags importantes
    for col in feature_cols[:8]:
        for lag in [1, 2, 3, 5]:
            df_feat[f'{col}_lag_{lag}'] = df_feat[col].shift(lag)
    
    # 2. Rolling statistics
    for col in feature_cols[:5]:
        for window in [5, 10, 20]:
            df_feat[f'{col}_ma_{window}'] = df_feat[col].rolling(window).mean()
            df_feat[f'{col}_std_{window}'] = df_feat[col].rolling(window).std()
    
    # 3. Ratios
    for i in range(min(3, len(feature_cols))):
        for j in range(i+1, min(5, len(feature_cols))):
            col1, col2 = feature_cols[i], feature_cols[j]
            df_feat[f'{col1}_{col2}_ratio'] = df_feat[col1] / (df_feat[col2] + 1e-8)
    
    # 4. Momentum
    for col in feature_cols[:3]:
        df_feat[f'{col}_roc_5'] = (df_feat[col] / df_feat[col].shift(5) - 1) * 100
        df_feat[f'{col}_roc_10'] = (df_feat[col] / df_feat[col].shift(10) - 1) * 100
    
    # 5. Target lags (históricos)
    if 'forward_return_1d' in df_feat.columns:
        for lag in [2, 3, 5, 10]:
            df_feat[f'target_lag_{lag}'] = df_feat['forward_return_1d'].shift(lag)
    
    # 6. Características temporales
    df_feat['day_of_year'] = df_feat['date_id'] % 252
    df_feat['day_sin'] = np.sin(2 * np.pi * df_feat['day_of_year'] / 252)
    df_feat['day_cos'] = np.cos(2 * np.pi * df_feat['day_of_year'] / 252)
    
    return df_feat

print("✅ Función de características definida")

In [ ]:
# Cargar y preparar datos
train_df = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
print(f"📊 Datos cargados: {train_df.shape}")

# Aplicar ingeniería de características
train_enhanced = create_features(train_df)
train_clean = train_enhanced.dropna()
print(f"📊 Datos procesados: {train_clean.shape}")

# Preparar características y target
feature_columns = [col for col in train_clean.columns 
                  if col not in ['date_id', 'forward_return_1d']]
X = train_clean[feature_columns]
y = train_clean['forward_return_1d']

print(f"🎯 Características disponibles: {len(feature_columns)}")

In [ ]:
# Selección de características
lgb_selector = lgb.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    verbose=-1
)
lgb_selector.fit(X, y)

selector = SelectFromModel(lgb_selector, prefit=True, max_features=40)
selected_features = X.columns[selector.get_support()].tolist()
X_selected = X[selected_features]

print(f"🎯 Características seleccionadas: {len(selected_features)}")

# Split temporal
split_point = int(len(X_selected) * 0.85)
X_train = X_selected.iloc[:split_point]
y_train = y.iloc[:split_point]
X_val = X_selected.iloc[split_point:]
y_val = y.iloc[split_point:]

print(f"📊 Train: {X_train.shape}, Validation: {X_val.shape}")

In [ ]:
# Entrenar modelos del ensemble
models = {}

# LightGBM
print("🤖 Entrenando LightGBM...")
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    verbose=-1
)
lgb_model.fit(X_train, y_train)
models['lgb'] = lgb_model

# XGBoost
print("🤖 Entrenando XGBoost...")
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train, y_train)
models['xgb'] = xgb_model

# CatBoost
print("🤖 Entrenando CatBoost...")
cat_model = cb.CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3,
    random_seed=42,
    verbose=False
)
cat_model.fit(X_train, y_train)
models['cat'] = cat_model

print("✅ Modelos entrenados")

In [ ]:
# Evaluar y calcular pesos del ensemble
model_scores = {}
for name, model in models.items():
    pred_val = model.predict(X_val)
    score = adjusted_sharpe_ratio(y_val, pred_val)
    model_scores[name] = score
    print(f"📊 {name}: {score:.6f}")

# Calcular pesos basados en performance
scores = list(model_scores.values())
min_score = min(scores)
adjusted_scores = [max(score - min_score + 0.001, 0.001) for score in scores]
total_score = sum(adjusted_scores)
weights = [score / total_score for score in adjusted_scores]

print(f"\n🏆 Pesos del ensemble: {[f'{w:.3f}' for w in weights]}")

# Test ensemble
def ensemble_predict(X, models, weights):
    predictions = []
    for i, (name, model) in enumerate(models.items()):
        pred = model.predict(X) * weights[i]
        predictions.append(pred)
    return np.sum(predictions, axis=0)

ensemble_pred = ensemble_predict(X_val, models, weights)
ensemble_score = adjusted_sharpe_ratio(y_val, ensemble_pred)
print(f"🏆 Ensemble Score: {ensemble_score:.6f}")

In [ ]:
# Función de predicción final
def predict(test_df):
    """
    Función de predicción para la API de Kaggle
    """
    try:
        # Aplicar ingeniería de características
        test_enhanced = create_features(test_df)
        
        # Seleccionar características
        available_features = [f for f in selected_features if f in test_enhanced.columns]
        missing_features = [f for f in selected_features if f not in test_enhanced.columns]
        
        if missing_features:
            for feature in missing_features:
                test_enhanced[feature] = 0.0
        
        test_features = test_enhanced[selected_features]
        
        # Manejar valores faltantes
        test_features = test_features.fillna(method='ffill')
        test_features = test_features.fillna(method='bfill')
        test_features = test_features.fillna(0.0)
        
        # Predicciones del ensemble
        predictions = ensemble_predict(test_features, models, weights)
        
        # Aplicar límites y suavizado
        predictions = np.clip(predictions * 0.8, -6.0, 6.0)
        
        return predictions
        
    except Exception as e:
        print(f"Error: {e}")
        return np.zeros(len(test_df))

print("✅ Función de predicción definida")

In [ ]:
# Test de la función de predicción
test_sample = train_df.tail(100)
test_predictions = predict(test_sample)
print(f"🧪 Test predictions: {len(test_predictions)} samples")
print(f"🧪 Range: [{test_predictions.min():.3f}, {test_predictions.max():.3f}]")
print(f"🧪 Mean: {test_predictions.mean():.3f}, Std: {test_predictions.std():.3f}")

In [ ]:
# Ejecutar evaluación de Kaggle
import kaggle_evaluation.hull_tactical_market_prediction as evaluation
evaluation.run(predict)